In [1]:
import cosmosis
import os

import numpy as np
from jax.tree_util import register_pytree_node
import sys
sys.path.append("background/jax_background")
sys.path.append("likelihood/pantheon")
import jax_background
import pantheon_jax

import jax
import jax_background2


/Users/jzuntz/src/cosmosis/env/lib/python3.11/site-packages/jax_cosmo/__init__.py:2: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound


In [2]:
pwd = os.getcwd()
override={
    ("runtime", "root"): pwd,    
    }
ini = cosmosis.Inifile("examples/pantheon.ini", )
ini.set("runtime", "root", pwd)
p = cosmosis.LikelihoodPipeline(ini)
v = p.start_vector()

# Do everything once to set up caching in consistency etc.
p.run_results(v)

block = p.build_starting_block(v)
p.run(block)
like = block.get_double("likelihoods", "pantheon_like")



Setting up module consistency
------------------------------

Setting up module astropy
--------------------------

Setting up module pantheon
---------------------------
Loading Pantheon data from /Users/jzuntz/src/cosmosis/cosmosis-standard-library/likelihood/pantheon/lcparam_DS17f.txt
Found 40 Pantheon supernovae (or bins if you used the binned data file)
Loading Pantheon covariance from /Users/jzuntz/src/cosmosis/cosmosis-standard-library/likelihood/pantheon/sys_DS17f.txt
Setup all pipeline modules


Parameter Priors
----------------
cosmological_parameters--omega_m  ~ U(0.15, 0.4)
cosmological_parameters--h0       ~ U(0.6, 0.8)
cosmological_parameters--w        ~ delta(-1.0)
cosmological_parameters--omega_b  ~ delta(0.04)
cosmological_parameters--omega_k  ~ U(-0.2, 0.2)
cosmological_parameters--a_s      ~ delta(2e-09)
cosmological_parameters--n_s      ~ delta(1.0)
cosmological_parameters--tau      ~ delta(0.08)
supernova_params--m               ~ U(-21.0, -18.0)

Consistency rela

In [3]:
def determine_used_values(block):
    nlog = block.get_log_count()
    logs = [block.get_log_entry(i) for i in range(nlog)]
    used = []
    for log in logs:
        if log[0] == "READ-OK" or log[0] == "READ-DEFAULT":
            sec = log[1]
            name = log[2]
            used.append((sec, name))
    return used


In [4]:
class SpecializedDataBlock:
    """
    A CosmoSIS Datablock that only holds data that
    gets used further down the pipeline.

    """

    def __init__(self, used_keys):
        # look through the block's log and determine what data is used later
        # in the pipeline and what is never read
        self.used_keys = set(used_keys)
        self.values = {}

    def initalize(self, pipeline, x):
        params = pipeline.varied_params

        for i, param in enumerate(params):
            section = param.section
            name = param.name
            value = x[i]
            self[section, name] = value

        for param in pipeline.fixed_params:
            section = param.section
            name = param.name
            value = param.start
            self[section, name] = value

    @staticmethod
    def _normalize_key(key):
        sec, name = key
        sec = sec.lower()
        name = name.lower()
        return (sec, name)
    
    def __setitem__(self, key, value):
        key = self._normalize_key(key)
        if key in self.used_keys:
            self.values[key] = value

    def log_access(self, *args, **kwargs):
        pass

    def get_double(self, section, name, default=None):
        if (section, name) not in self.values and default is not None:
            self[section, name] = default
        return self[section, name]
    
    def put_double(self, section, name, value):
        self[section, name] = value

    def put_metadata(self, *args, **kwargs):
        pass

    # TODO: Handle grids and other get/set methods

    def has_value(self, section, name):
        key = self._normalize_key((section, name))
        return key in self.used_keys and key in self.values
    
    def __getitem__(self, key):
        key = self._normalize_key(key)
        if key in self.used_keys:
            return self.values[key]
        else:
            raise KeyError(f"Key {key} not available in pipeline:" + str(self.used_keys))
        
    def has_key(self, key):
        return key in self.used_keys


def flatten_block(block):
    children = list(block.values.values())
    aux_data = [list(block.values.keys()), block.used_keys]
    return children, aux_data

def unflatten_block(aux_data, children):
    keys, used_keys = aux_data
    block = SpecializedDataBlock(used_keys)
    block.values = dict(zip(keys, children))
    return block


used_keys = determine_used_values(block)
special_block = SpecializedDataBlock(used_keys)

register_pytree_node(SpecializedDataBlock, flatten_block, unflatten_block)

In [5]:
special_block.initalize(p, p.start_vector())
p.run(special_block)

Consistency relation input parameters: omega_m, omega_b, h0, omega_k, nnu, TCMB

Using cached assumptions:  omega_nu=0
Using cached relation ommh2 = omega_m*h0*h0
Using cached relation ombh2 = omega_b*h0*h0
Using cached relation omnuh2 = omega_nu*h0*h0
Using cached relation omch2 = ommh2-ombh2-omnuh2
Using cached relation baryon_fraction = omega_b/omega_m
Using cached relation omega_c = omega_m-omega_b-omega_nu
Using cached relation hubble = h0*100
Using cached relation omega_lambda = 1-omega_m-omega_k
Using cached relation K = -hubble*hubble*omega_k/299792.458/299792.458
Using cached relation mnu = omnuh2 / ((nnu / 3.0) ** 0.75 / 94.06410581217612 * (TCMB/2.7255)**3)
Using cached relation omlamh2 = omega_lambda*h0*h0


True

In [6]:
ini = cosmosis.Inifile("examples/pantheon_nuts.ini")
ini.set("runtime", "root", pwd)
pipe = cosmosis.LikelihoodPipeline(ini)


Setting up module jax_background
---------------------------------

Setting up module pantheon_jax
-------------------------------
Loading Pantheon data from /Users/jzuntz/src/cosmosis/cosmosis-standard-library/likelihood/pantheon/lcparam_DS17f.txt
Found 40 Pantheon supernovae (or bins if you used the binned data file)
Pantheon redshift range: 0.014 to 1.6123
Loading Pantheon covariance from /Users/jzuntz/src/cosmosis/cosmosis-standard-library/likelihood/pantheon/sys_DS17f.txt
Setup all pipeline modules


Parameter Priors
----------------
cosmological_parameters--omega_m  ~ U(0.15, 0.4)
cosmological_parameters--h0       ~ U(0.6, 0.8)
cosmological_parameters--w        ~ delta(-1.0)
cosmological_parameters--omega_k  ~ U(-0.2, 0.2)
supernova_params--m               ~ U(-21.0, -18.0)



In [7]:
function0 = jax_background2.execute

inv_cov = jax.numpy.array(pipe.modules[1].data.inv_cov.copy())
data_vector = jax.numpy.array(pipe.modules[1].data.data_y.copy())
data_z = jax.numpy.array(pipe.modules[1].data.data_x.copy())

def function1(block, config):
    data_vector = config["data_vector"]
    inv_cov = config["inv_cov"]
    data_z = config["data_z"]
    jax.lax.stop_gradient(config)

    # data_vector, inv_cov, data_z = config
    z = block["distances", "z"][1:]
    mu = block["distances", "mu"][1:]
    m = block["supernova_params", "m"]

    # interpolate to data z
    sample_mu =  jax.numpy.interp(data_z, z, mu)
    model_mu = sample_mu + m
    delta = data_vector - model_mu
    like = -0.5 * jax.numpy.dot(delta, jax.numpy.dot(inv_cov, delta))
    block["likelihoods", "pantheon_like"] = like
    return block


# function0 = jax.jit(jax_background2.execute, static_argnames=["config"])

In [ ]:
class Config:
    def __init__(self, **kwargs):
        self.types_info = []
        self.values = {}
        
        for (key, value) in kwargs.items():
            is_array = isinstance(value, (jax.Array, np.ndarray))
            self.types_info.append((key, is_array))
            self.values[key] = value

    def __getitem__(self, key):
        return self.values[key]


def _config_flatten(obj):
    # the children array consists of all array-valued attributes
    children = []
    non_children = {}

    for key, is_array in obj.types_info:
        val = obj.values[key]
        if is_array:
            children.append(val)
        else:
            non_children[key] = val

    aux_data = {
        "types_info": obj.types_info,
        "non_children": non_children,
    }
    return tuple(children), aux_data


def _config_unflatten(aux_data, children):
    types_info = aux_data["types_info"]
    non_children = aux_data["non_children"]
    obj = Config()
    obj.types_info = types_info
    obj.values = {}
    
    child_idx = 0
    for key, is_array in types_info:
        if is_array:
            obj.values[key] = children[child_idx]
            child_idx += 1
        else:
            obj.values[key] = non_children[key]
    return obj

config0 = Config(nz=100, zmax=2.0)

register_pytree_node(Config, _config_flatten, _config_unflatten)

b = SpecializedDataBlock(
    [('cosmological_parameters', "omega_m"),
    ('cosmological_parameters', "h0"),
    ('cosmological_parameters', "w"),
    ('cosmological_parameters', "omega_k"),
    ('supernova_params', 'm'),
    ('distances', 'mu'),
    ('distances', 'z'),
    ('likelihoods', 'pantheon_like')
    ]
)
b["cosmological_parameters", "omega_m"] = 0.3
b["cosmological_parameters", "h0"] = 0.7
b["cosmological_parameters", "w"] = -1.0
b["cosmological_parameters", "omega_k"] = 0.0
b["supernova_params", "m"] = -19.3

config1 = Config(data_vector=data_vector, inv_cov=inv_cov, data_z=data_z)
b = jax.jit(function0)(b, config0)
b = jax.jit(function1)(b, config1)
print(b["likelihoods", "pantheon_like"])

def wrap1(v, config):
    b = SpecializedDataBlock(
        [('cosmological_parameters', "omega_m"),
        ('cosmological_parameters', "h0"),
        ('cosmological_parameters', "w"),
        ('cosmological_parameters', "omega_k"),
        ('supernova_params', 'm'),
        ('distances', 'mu'),
        ('distances', 'z'),
        ('likelihoods', 'pantheon_like')
        ]
    )
    b["cosmological_parameters", "omega_m"] = v[0]
    b["cosmological_parameters", "h0"] = v[1]
    b["cosmological_parameters", "w"] = -v[2]
    b["cosmological_parameters", "omega_k"] = v[3]
    b = function0(b, config)
    return b['distances', 'mu']

v0 = jax.numpy.array([0.3, 0.7, -1.0, 0.0])

grad0 = jax.jacobian(wrap1)(v0, config0)


/Users/jzuntz/src/cosmosis/env/lib/python3.11/site-packages/jax/_src/numpy/array_methods.py:122: UserWarning: Explicitly requested dtype <class 'jax.numpy.int64'> requested in astype is not available, and will be truncated to dtype int32. To enable more dtypes, set the jax_enable_x64 configuration option or the JAX_ENABLE_X64 shell environment variable. See https://github.com/jax-ml/jax#current-gotchas for more.
  return lax_numpy.astype(self, dtype, copy=copy, device=device)


-3916.2512


In [24]:
def specialized_block_factory(pipeline, block):
    used_keys = determine_used_values(block)
    varied_params = [(param.section, param.name) for param in pipeline.varied_params]
    fixed_params = [(param.section, param.name, param.start) for param in pipeline.fixed_params]
    print(varied_params)
    def factory(v, varied_params=varied_params, fixed_params=fixed_params, used_keys=used_keys):
        b = SpecializedDataBlock(used_keys)
        params = varied_params

        for i, param in enumerate(params):
            section, name = param
            value = v[i]
            b[section, name] = value

        for param in fixed_params:
            section, name, value = param
            b[section, name] = value
        return b
    return jax.jit(factory, static_argnames=["varied_params", "fixed_params", "used_keys"])

specialized_factory = specialized_block_factory(p, block)


[('cosmological_parameters', 'omega_m'), ('cosmological_parameters', 'h0'), ('cosmological_parameters', 'omega_k'), ('supernova_params', 'm')]


In [25]:
v0 = jax.numpy.array([0.3, 0.7, 0.0, -19.3])

specialized_block = specialized_factory(v0)

In [ ]:

def abstract_syntax_tree_rewriter(function):
    # go through the function and replace any return statements
    # that return 0 with return statements that return the block
    # any statements that return non-zero literal values should
    # be replaced with statements that raise an error.
    # statements that return non-literal values should
    # be replaced with tests of whether the value is zero or not
    # and return the block if zero, raise error if not.
    # if it is already returning the block, do nothing.


In [ ]:

def like(v, configs):
    jax.lax.stop_gradient(configs)
    b = SpecializedDataBlock(
        [('cosmological_parameters', "omega_m"),
        ('cosmological_parameters', "h0"),
        ('cosmological_parameters', "w"),
        ('cosmological_parameters', "omega_k"),
        ('supernova_params', 'm'),
        ('distances', 'mu'),
        ('distances', 'z'),
        ('likelihoods', 'pantheon_like'),
        ]
    )
    b["cosmological_parameters", "omega_m"] = v[0]
    b["cosmological_parameters", "h0"] = v[1]
    b["cosmological_parameters", "w"] = v[2]
    b["cosmological_parameters", "omega_k"] = v[3]
    b["supernova_params", "m"] = v[4]
    b = function0(b, configs[0])
    b = function1(b, configs[1])
    return b["likelihoods", "pantheon_like"]


v0 = jax.numpy.array([0.3, 0.7, -1.0, 0.0, -19.3])
likej = jax.jit(like)
like0 = likej(v0, (config0, config1))
dlike0 = jax.jit(jax.grad(like))
grad0 = dlike0(v0, (config0, config1))


/Users/jzuntz/src/cosmosis/env/lib/python3.11/site-packages/jax/_src/numpy/array_methods.py:122: UserWarning: Explicitly requested dtype <class 'jax.numpy.int64'> requested in astype is not available, and will be truncated to dtype int32. To enable more dtypes, set the jax_enable_x64 configuration option or the JAX_ENABLE_X64 shell environment variable. See https://github.com/jax-ml/jax#current-gotchas for more.
  return lax_numpy.astype(self, dtype, copy=copy, device=device)


In [22]:
like0, grad0

(Array(-3916.2512, dtype=float32),
 Array([  3932.5815,  32158.941 ,   1896.5001,   2284.5383, -10366.816 ],      dtype=float32))

In [ ]:
# Steps to building a JAX-able CosmoSIS likelihood function
# 0. Run the pipeline once in normal mode to determine the keys we need to keep in the new specialized data block
# 1. Go through each step in the pipeline and convert the configuration information to a JAX pytree-compatible object (e.g. Config above)
# 2. Convert the CosmoSIS Datablock to a SpecializedDataBlock that only holds data that is used later in the pipeline
# 3. Rewrite all the functions in the pipeline to return the SpecializedDataBlock that they accept as input instead of just modifying it in place
# 4. Write a function factory to convert from a parameter vector to the SpecializedDataBlock object based on the pipeline information
# 5 Write a combined function factory 